# Open Images V7 — Download Subset

This notebook runs the local download of an Open Images V7 subset on Colab
instead of a developer's machine. Local runs hit pandas memory issues when
FiftyOne loads the full V7 split index into RAM; Colab has enough headroom
to handle that step.

This notebook does NOT:
- Convert annotations to YOLO format.
- Train any model.
- Use DVC or MLflow.

Outputs the COCO export zip to Drive at
`MyDrive/iaa-table-assistant/raw_datasets/open_images/` so the rest of the
pipeline (running locally or on Colab) can pick it up via
`setup_colab_raw_datasets.py`.

## 1. Environment Setup

In [ ]:
REPO_URL = "https://github.com/LucasGVallejos/iaa-visual-table-assistant"
WORKSPACE_DIR = "/content"
PROJECT_DIR = "iaa-visual-table-assistant"
PROJECT_PATH = f"{WORKSPACE_DIR}/{PROJECT_DIR}"

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/iaa-table-assistant/raw_datasets/open_images"

In [ ]:
!pip install -q fiftyone

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Clone the Repository

In [ ]:
import os

%cd {WORKSPACE_DIR}

if not os.path.exists(PROJECT_PATH):
    !git clone {REPO_URL} {PROJECT_PATH}
else:
    print("Repository already exists. Pulling latest changes...")
    !git -C {PROJECT_PATH} pull

%cd {PROJECT_PATH}

In [ ]:
!pip install -q -r requirements.txt

## 4. Run the Download Script

This step takes a while (per-class downloads, then dedupe + COCO export +
zip). Keep the tab active so Colab does not disconnect for inactivity.

In [ ]:
!python -m src.data.raw_setup.download_open_images_subset

## 5. Persist the Zip on Drive

Colab storage is wiped when the runtime ends. Copying the export zip to
Drive ensures the artifact survives the session and is consumable by the
next stage of the pipeline.

In [ ]:
import shutil
from pathlib import Path

ZIP_LOCAL = Path("local_data/raw_datasets/open_images_table_objects_v1_coco.zip")
DRIVE_DIR = Path(DRIVE_OUTPUT_DIR)

if not ZIP_LOCAL.exists():
    raise FileNotFoundError(f"Expected zip not found at {ZIP_LOCAL}")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
destination = DRIVE_DIR / ZIP_LOCAL.name

print(f"Copying {ZIP_LOCAL} -> {destination}")
shutil.copy2(ZIP_LOCAL, destination)

size_mb = destination.stat().st_size / (1024 * 1024)
print(f"Done. {destination} ({size_mb:.1f} MB)")